# Vaani Paired Telephony Pilot

This notebook builds the controlled CallWhisper-8k benchmark artifact. It selects 500 speaker-unique Vaani Benchmark utterances and creates five paired versions of each: original, bandwidth-limited 8 kHz, and bandwidth-limited 8 kHz followed by G.711 A-law, G.711 mu-law, or GSM-FR.

It does **not** evaluate models or train anything. The output is a validated, revision-pinned benchmark pilot that can be inspected before GPU inference.

## One-Time Prerequisite

1. Open [ARTPARK-IISc/Vaani-Benchmark-V1.0](https://huggingface.co/datasets/ARTPARK-IISc/Vaani-Benchmark-V1.0), sign in, and accept the dataset conditions.
2. Create a read-only Hugging Face token.
3. In Colab, open **Secrets**, add `HF_TOKEN`, and enable notebook access.

The dataset is gated. The notebook will stop with a direct explanation if access is missing.

In [ ]:
# Environment setup: mount Drive, clone safely, and install only benchmark dependencies.
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
REPO_DIR = Path('/content/CallWhisper-8k')

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets>=4.0,<5', 'huggingface_hub>=0.34,<2',
    'pandas>=2.0', 'tqdm>=4.66', 'tabulate>=0.9',
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)

commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository:', REPO_DIR)
print('Repository commit:', commit)
subprocess.run(['ffmpeg', '-version'], check=True)

In [ ]:
# Frozen pilot configuration. Do not change this after model results exist.
import json
import platform
from importlib.metadata import version

from google.colab import userdata
from huggingface_hub import HfApi, get_token, login

DATASET_ID = 'ARTPARK-IISc/Vaani-Benchmark-V1.0'
DATASET_REVISION = '1bf019521d12d742178acc32bf2a42f81cf7c8ef'
SPLIT = 'test'
EXPECTED_DATASET_ROWS = 5050
PILOT_SIZE = 500
SEED = 0
CONDITIONS = (
    'original', 'bandlimit_8k',
    'bandlimit_8k_g711_alaw', 'bandlimit_8k_g711_mulaw', 'bandlimit_8k_gsm_fr',
)

# Only use these if the live schema cannot be inferred. Values are column names.
COLUMN_OVERRIDES = {
    'audio': None,
    'source_id': None,
    'speaker_id': None,
    'gender': None,
    'state': None,
    'district': None,
    'references': None,  # Example: ['transcription_1', 'transcription_2', 'transcription_3']
}

WORK_ROOT = Path('/content/vaani_paired_pilot_v2')
DRIVE_OUTPUT = DRIVE_PROJECT_DIR / 'results/vaani_paired_pilot_v2'
ARTIFACTS_DIR = WORK_ROOT / 'artifacts'
SOURCE_DIR = WORK_ROOT / 'source_audio'
PAIRED_AUDIO_DIR = WORK_ROOT / 'paired_audio'
ARCHIVE_DIR = DRIVE_OUTPUT / 'archives'
for directory in (ARTIFACTS_DIR, SOURCE_DIR, PAIRED_AUDIO_DIR, DRIVE_OUTPUT, ARCHIVE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

try:
    secret = userdata.get('HF_TOKEN')
except Exception:
    secret = None
if secret:
    login(token=secret, add_to_git_credential=False)
token = get_token()
if not token:
    raise RuntimeError('HF_TOKEN is missing. Add it to Colab Secrets and enable notebook access.')

protocol = {
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'split': SPLIT,
    'expected_dataset_rows': EXPECTED_DATASET_ROWS,
    'pilot_size': PILOT_SIZE,
    'seed': SEED,
    'conditions': CONDITIONS,
    'channel_semantics': 'bandlimit first, then codec for every codec condition',
    'selection': 'one deterministic row per speaker, round-robin over gender and state',
    'training_allowed': False,
    'repo_commit': commit,
}
(DRIVE_OUTPUT / 'protocol_config.json').write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
versions = {name: version(name) for name in ('datasets', 'huggingface_hub', 'pandas', 'tqdm')}
versions.update({'python': platform.python_version(), 'repo_commit': commit})
(DRIVE_OUTPUT / 'package_versions.json').write_text(
    json.dumps(versions, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(protocol, indent=2))

In [ ]:
# Verify gated access and load the exact pinned revision without decoding audio.
from datasets import Audio, load_dataset

api = HfApi(token=token)
try:
    dataset_info = api.dataset_info(DATASET_ID, revision=DATASET_REVISION)
except Exception as exc:
    raise RuntimeError(
        'Vaani access failed. Accept the dataset conditions on Hugging Face, then check HF_TOKEN.'
    ) from exc
if dataset_info.sha != DATASET_REVISION:
    raise RuntimeError(f'Dataset revision mismatch: {dataset_info.sha}')

print('Downloading pinned Vaani revision. The first run downloads several GB.')
dataset = load_dataset(
    DATASET_ID, split=SPLIT, revision=DATASET_REVISION, token=token,
    cache_dir='/content/huggingface_cache',
)
audio_feature_columns = [
    name for name, feature in dataset.features.items() if isinstance(feature, Audio)
]
if COLUMN_OVERRIDES['audio']:
    audio_column = COLUMN_OVERRIDES['audio']
elif len(audio_feature_columns) == 1:
    audio_column = audio_feature_columns[0]
else:
    raise RuntimeError(f'Expected one Audio column, found: {audio_feature_columns}')
dataset = dataset.cast_column(audio_column, Audio(decode=False))
if len(dataset) != EXPECTED_DATASET_ROWS:
    raise RuntimeError(f'Pinned dataset row count changed: {len(dataset)} != {EXPECTED_DATASET_ROWS}')
print('Rows:', len(dataset))
print('Pinned revision:', dataset_info.sha)
print('Columns:', dataset.column_names)
print('Features:', dataset.features)

In [ ]:
# Infer and freeze the dataset schema used by this run.
import re

preview = dataset[0]
columns = dataset.column_names

def choose_column(role, candidates, required=True):
    override = COLUMN_OVERRIDES.get(role)
    if override:
        if override not in columns:
            raise RuntimeError(f'Override for {role} is not a dataset column: {override}')
        return override
    lowered = {name.lower(): name for name in columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    if required:
        raise RuntimeError(f'Could not infer {role}. Available columns: {columns}')
    return None

source_id_column = choose_column(
    'source_id', ('segment_id', 'audio_id', 'utterance_id', 'sample_id', 'id'), required=False
)
speaker_column = choose_column(
    'speaker_id', ('speaker_id', 'speaker', 'user_id', 'respondent_id', 'participant_id')
)
gender_column = choose_column('gender', ('gender', 'sex'))
state_column = choose_column('state', ('state', 'state_name'))
district_column = choose_column('district', ('district', 'district_name'))

reference_override = COLUMN_OVERRIDES.get('references')
if reference_override:
    reference_columns = list(reference_override)
else:
    reference_candidates = [
        name for name in columns
        if re.search(r'(transcri|transcript|reference|ref[_-]?text|annotation)', name, re.I)
        and isinstance(preview.get(name), str)
    ]
    numbered_references = [
        name for name in reference_candidates if re.search(r'[123]$', name)
    ]
    reference_columns = sorted(
        numbered_references if len(numbered_references) == 3 else reference_candidates
    )
list_reference_columns = [
    name for name in columns
    if re.search(r'(transcri|transcript|reference|annotation)', name, re.I)
    and isinstance(preview.get(name), (list, tuple))
]
if len(reference_columns) < 3 and len(list_reference_columns) == 1:
    reference_columns = [list_reference_columns[0]]
if len(reference_columns) != 3 and not (
    len(reference_columns) == 1 and isinstance(preview.get(reference_columns[0]), (list, tuple))
):
    raise RuntimeError(
        f'Expected three reference columns or one list-valued reference column; inferred {reference_columns}. '
        'Set COLUMN_OVERRIDES[\'references\'] explicitly and rerun from the config cell.'
    )

schema = {
    'audio': audio_column, 'source_id': source_id_column, 'speaker_id': speaker_column,
    'gender': gender_column, 'state': state_column, 'district': district_column,
    'references': reference_columns,
}
(DRIVE_OUTPUT / 'dataset_schema.json').write_text(
    json.dumps(schema, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(schema, ensure_ascii=False, indent=2))

In [ ]:
# Build the full metadata inventory and deterministic 500-speaker pilot.
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from callwhisper.datasets.paired_telephony import (
    deterministic_stratified_sample, normalize_group, sample_key, stable_key,
)

metadata_dataset = dataset.remove_columns(audio_column)
inventory_rows = []
for index, row in enumerate(tqdm(metadata_dataset, total=len(metadata_dataset), desc='Inventory')):
    if len(reference_columns) == 1 and isinstance(row.get(reference_columns[0]), (list, tuple)):
        references = [str(value or '').strip() for value in row[reference_columns[0]]]
    else:
        references = [str(row.get(name) or '').strip() for name in reference_columns]
    references = references[:3]
    while len(references) < 3:
        references.append('')
    raw_source_id = row.get(source_id_column) if source_id_column else None
    source_id = str(raw_source_id) if raw_source_id not in (None, '') else f'row-{index:06d}'
    inventory_rows.append({
        'dataset_index': index,
        'source_id': source_id,
        'sample_key': sample_key(DATASET_ID, source_id),
        'speaker_id': normalize_group(row.get(speaker_column)),
        'gender': normalize_group(row.get(gender_column)),
        'state': normalize_group(row.get(state_column)),
        'district': normalize_group(row.get(district_column)),
        'reference_1': references[0],
        'reference_2': references[1],
        'reference_3': references[2],
    })

inventory_df = pd.DataFrame(inventory_rows)
eligible_df = inventory_df[
    inventory_df[['reference_1', 'reference_2', 'reference_3']].ne('').all(axis=1)
    & inventory_df['speaker_id'].ne('unknown')
].copy()
if eligible_df['sample_key'].duplicated().any():
    raise RuntimeError('Duplicate sample keys detected')
pilot_rows = deterministic_stratified_sample(
    eligible_df.to_dict('records'), PILOT_SIZE, SEED,
    speaker_column='speaker_id', stratum_columns=('gender', 'state'),
)
pilot_df = pd.DataFrame(pilot_rows).sort_values('sample_key').reset_index(drop=True)
assert len(pilot_df) == PILOT_SIZE
assert pilot_df['speaker_id'].nunique() == PILOT_SIZE

inventory_path = DRIVE_OUTPUT / 'vaani_full_inventory.csv'
pilot_path = DRIVE_OUTPUT / 'vaani_pilot_500.csv'
inventory_df.to_csv(inventory_path, index=False)
pilot_df.to_csv(pilot_path, index=False)
selection_summary = {
    'inventory_rows': int(len(inventory_df)),
    'eligible_rows': int(len(eligible_df)),
    'pilot_rows': int(len(pilot_df)),
    'unique_speakers': int(pilot_df['speaker_id'].nunique()),
    'sample_key_set_sha256': stable_key(*sorted(pilot_df['sample_key'])),
    'gender_counts': pilot_df['gender'].value_counts().sort_index().to_dict(),
    'state_counts': pilot_df['state'].value_counts().sort_index().to_dict(),
}
(DRIVE_OUTPUT / 'pilot_selection_summary.json').write_text(
    json.dumps(selection_summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
display(pilot_df.groupby(['gender', 'state']).size().rename('files').reset_index())
print('Full rows:', len(inventory_df))
print('Eligible rows with three references and known speaker:', len(eligible_df))
print('Pilot rows / unique speakers:', len(pilot_df), pilot_df['speaker_id'].nunique())
print('Saved:', inventory_path, pilot_path)

In [ ]:
# Archive helpers make every completed stage restartable from Drive.
import tarfile

def safe_extract(archive_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive_path, 'r:gz') as archive:
        members = archive.getmembers()
        for member in members:
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing archive link: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe archive path: {member.name}')
        for member in tqdm(members, desc=f'Restoring {archive_path.name}'):
            archive.extract(member, destination, filter='data')

def copy_with_progress(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    total = source.stat().st_size
    with source.open('rb') as input_handle, temporary.open('wb') as output_handle, tqdm(
        total=total, unit='B', unit_scale=True, desc=f'Copying {destination.name}'
    ) as progress:
        while chunk := input_handle.read(8 * 1024 * 1024):
            output_handle.write(chunk)
            progress.update(len(chunk))
    temporary.replace(destination)

def archive_directory(source_dir, archive_path, arcname):
    temporary = Path('/content') / f'{archive_path.name}.part'
    if temporary.exists():
        temporary.unlink()
    files = sorted(path for path in source_dir.rglob('*') if path.is_file())
    with tarfile.open(temporary, 'w:gz') as archive:
        for path in tqdm(files, desc=f'Archiving {arcname}'):
            archive.add(path, arcname=str(Path(arcname) / path.relative_to(source_dir)))
    copy_with_progress(temporary, archive_path)
    temporary.unlink()
    print('Saved archive:', archive_path, 'GB=', round(archive_path.stat().st_size / 1e9, 3))

def restore_if_available(archive_path):
    if archive_path.exists():
        print('Restoring completed stage:', archive_path)
        safe_extract(archive_path, WORK_ROOT)
        return True
    return False

In [ ]:
# Export the 500 compressed source clips without TorchCodec decoding.
from callwhisper.datasets.paired_telephony import sha256_file

source_archive = ARCHIVE_DIR / 'source_audio.tar.gz'
restore_if_available(source_archive)
source_records = []
for row in tqdm(pilot_df.to_dict('records'), desc='Exporting source audio'):
    payload = dataset[int(row['dataset_index'])][audio_column]
    original_name = str(payload.get('path') or '') if isinstance(payload, dict) else ''
    suffix = Path(original_name).suffix.lower() or '.audio'
    destination = SOURCE_DIR / f"{row['sample_key']}{suffix}"
    if not destination.exists():
        if isinstance(payload, dict) and payload.get('bytes') is not None:
            destination.write_bytes(payload['bytes'])
        elif isinstance(payload, dict) and payload.get('path') and Path(payload['path']).exists():
            shutil.copy2(payload['path'], destination)
        else:
            raise RuntimeError(f'Audio payload has neither bytes nor a local path: {payload}')
    source_records.append({
        'sample_key': row['sample_key'],
        'source_audio_path': str(destination.relative_to(WORK_ROOT)),
        'source_sha256': sha256_file(destination),
    })
source_df = pd.DataFrame(source_records)
assert len(source_df) == PILOT_SIZE
source_df.to_csv(DRIVE_OUTPUT / 'source_audio_manifest.csv', index=False)
if not source_archive.exists():
    archive_directory(SOURCE_DIR, source_archive, 'source_audio')
print('Source clips ready:', len(list(SOURCE_DIR.glob('*'))))

In [ ]:
# Generate and checkpoint each paired channel condition. Expect roughly 30-75 minutes.
from callwhisper.datasets.paired_telephony import (
    ffmpeg_version, probe_audio, transform_audio, validate_codec_support,
)

print(ffmpeg_version())
print('Codec support:', validate_codec_support(CONDITIONS))
source_lookup = source_df.set_index('sample_key').to_dict('index')
all_transform_rows = []
for condition in CONDITIONS:
    condition_dir = PAIRED_AUDIO_DIR / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    condition_archive = ARCHIVE_DIR / f'{condition}.tar.gz'
    restore_if_available(condition_archive)
    condition_rows = []
    for row in tqdm(pilot_df.to_dict('records'), desc=f'Building {condition}'):
        source_path = WORK_ROOT / source_lookup[row['sample_key']]['source_audio_path']
        output_path = condition_dir / f"{row['sample_key']}.wav"
        if output_path.exists():
            output_probe = probe_audio(output_path)
            source_probe = probe_audio(source_path)
            metadata = {
                'condition': condition, 'audio_path': str(output_path),
                'source_sha256': sha256_file(source_path),
                'output_sha256': sha256_file(output_path),
                'source_duration_s': source_probe['duration_s'],
                'duration_s': output_probe['duration_s'],
                'duration_delta_s': abs(output_probe['duration_s'] - source_probe['duration_s']),
                'sample_rate_hz': output_probe['sample_rate_hz'],
                'channels': output_probe['channels'],
            }
        else:
            metadata = transform_audio(source_path, output_path, condition)
        condition_rows.append({
            **row, **metadata,
            'audio_path': str(output_path.relative_to(WORK_ROOT)),
            'dataset_revision': DATASET_REVISION,
        })
    condition_df = pd.DataFrame(condition_rows)
    condition_manifest = DRIVE_OUTPUT / f'vaani_pilot_500_{condition}.csv'
    condition_df.to_csv(condition_manifest, index=False)
    all_transform_rows.extend(condition_rows)
    if not condition_archive.exists():
        archive_directory(condition_dir, condition_archive, f'paired_audio/{condition}')
    print('Completed:', condition, len(condition_df), 'files')

paired_df = pd.DataFrame(all_transform_rows)
paired_df.to_csv(DRIVE_OUTPUT / 'vaani_pilot_500_all_conditions.csv', index=False)

In [ ]:
# Validate the complete paired matrix and save the final audit.
counts = paired_df.groupby('sample_key')['condition'].nunique()
duration_tolerance = paired_df['source_duration_s'].mul(0.01).clip(lower=0.05)
duration_violations = paired_df['duration_delta_s'] > duration_tolerance
validation = {
    'source_rows': int(len(pilot_df)),
    'paired_rows': int(len(paired_df)),
    'expected_paired_rows': int(PILOT_SIZE * len(CONDITIONS)),
    'unique_speakers': int(pilot_df['speaker_id'].nunique()),
    'samples_with_all_conditions': int((counts == len(CONDITIONS)).sum()),
    'sample_rates_hz': sorted(int(value) for value in paired_df['sample_rate_hz'].unique()),
    'channels': sorted(int(value) for value in paired_df['channels'].unique()),
    'max_duration_delta_s': float(paired_df['duration_delta_s'].max()),
    'duration_tolerance_violations': int(duration_violations.sum()),
    'missing_output_files': int(sum(
        not (WORK_ROOT / path).exists() for path in paired_df['audio_path']
    )),
    'dataset_revision': DATASET_REVISION,
    'repo_commit': commit,
}
assert validation['paired_rows'] == validation['expected_paired_rows']
assert validation['unique_speakers'] == PILOT_SIZE
assert validation['samples_with_all_conditions'] == PILOT_SIZE
assert validation['sample_rates_hz'] == [16000]
assert validation['channels'] == [1]
assert validation['missing_output_files'] == 0
assert validation['duration_tolerance_violations'] == 0
validation_path = DRIVE_OUTPUT / 'validation_summary.json'
validation_path.write_text(
    json.dumps(validation, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print(json.dumps(validation, indent=2))
print('All paired benchmark artifacts are saved under:', DRIVE_OUTPUT)

In [ ]:
# Mandatory listening sanity check: hear the same utterance under every condition.
from IPython.display import Audio, Markdown, display

for key in pilot_df['sample_key'].head(3):
    display(Markdown(f'### Sample `{key}`'))
    row = pilot_df[pilot_df['sample_key'] == key].iloc[0]
    print('References:')
    print('1.', row['reference_1'])
    print('2.', row['reference_2'])
    print('3.', row['reference_3'])
    for condition in CONDITIONS:
        print(condition)
        display(Audio(str(PAIRED_AUDIO_DIR / condition / f'{key}.wav')))

## Stop Here

Do not train or launch the full model matrix yet. First confirm that:

- `validation_summary.json` passes every assertion;
- the three displayed examples sound like the same utterance under progressively different channels;
- all three references are legitimate alternatives;
- Drive contains the source archive, five condition archives, manifests, schema, protocol, and package versions.

Bring `validation_summary.json`, `dataset_schema.json`, and `vaani_pilot_500.csv` back to the repository. After review, notebook 12 will run ARTPARK and Adalat Whisper-small on this frozen pilot.